In [1]:
import requests

# Toronto Open Data is stored in a CKAN instance. It's APIs are documented here:
# https://docs.ckan.org/en/latest/api/

# To hit our API, you'll be making requests to:
base_url = "https://ckan0.cf.opendata.inter.prod-toronto.ca"

# Datasets are called "packages". Each package can contain many "resources"
# To retrieve the metadata for this package and its resources, use the package name in this page's URL:
url = base_url + "/api/3/action/package_show"
params = { "id": "bike-share-toronto-ridership-data"}
package = requests.get(url, params = params).json()

# To get resource data:
for idx, resource in enumerate(package["result"]["resources"]):

       # To get metadata for non datastore_active resources:
       if not resource["datastore_active"]:
           url = base_url + "/api/3/action/resource_show?id=" + resource["id"]
           resource_metadata = requests.get(url).json()
           print(resource_metadata)
           # From here, you can use the "url" attribute to download this file


{'help': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/api/3/action/help_show?name=resource_show', 'success': True, 'result': {'cache_last_updated': None, 'cache_url': None, 'created': '2019-07-23T16:40:47.062116', 'datastore_active': False, 'format': 'XLSX', 'hash': '', 'id': 'ad78f9f7-d4f2-42a1-9a1c-61dbc83f84c8', 'is_datastore_cache_file': False, 'is_preview': 'False', 'last_modified': '2022-04-12T14:38:55', 'metadata_modified': '2022-04-12T18:30:10.592092', 'mimetype': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet', 'mimetype_inner': None, 'name': 'bikeshare-ridership-readme', 'package_id': '7e876c24-177c-4605-9cef-e50dd74c617f', 'position': 0, 'resource_type': None, 'revision_id': '890aa1da-c8d0-4535-a719-14786ce56443', 'size': 16625, 'state': 'active', 'url': 'https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/7e876c24-177c-4605-9cef-e50dd74c617f/resource/ad78f9f7-d4f2-42a1-9a1c-61dbc83f84c8/download/bikeshare-ridership-readme.xlsx', 'url_type': 'uplo

In [ ]:
import zipfile
from pathlib import Path

RAW_DATA_DIR = Path("raw_data")
RAW_DATA_DIR.mkdir(exist_ok=True)

resources = [r for r in package["result"]["resources"] if r["name"] != "bikeshare-ridership-readme"]
[(r["name"], r["format"]) for r in resources]

[('bikeshare-ridership-2014-2015', 'XLSX'),
 ('bikeshare-ridership-2016', 'XLSX'),
 ('bikeshare-ridership-2017.zip', 'ZIP'),
 ('bikeshare-ridership-2018.zip', 'ZIP'),
 ('bikeshare-ridership-2019.zip', 'ZIP'),
 ('bikeshare-ridership-2020.zip', 'ZIP'),
 ('bikeshare-ridership-2021.zip', 'ZIP'),
 ('bikeshare-ridership-2022.zip', 'ZIP'),
 ('bikeshare-ridership-2023.zip', 'ZIP'),
 ('bikeshare-ridership-2024.zip', 'ZIP'),
 ('bikeshare-ridership-2025.zip', 'ZIP'),
 ('bikeshare-ridership-2026.zip', 'ZIP')]

In [7]:
def download_resource(resource, dest_dir=RAW_DATA_DIR, chunk_size=1 << 20):
    url = resource["url"]
    dest_path = dest_dir / url.split("/")[-1]
    if dest_path.exists():
        print(f"skip (already downloaded): {dest_path.name}")
        return dest_path

    print(f"downloading {dest_path.name} ...")
    with requests.get(url, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                f.write(chunk)
    return dest_path


def extract_if_zip(path, dest_dir=RAW_DATA_DIR):
    if path.suffix.lower() != ".zip":
        return path
    extract_dir = dest_dir / path.stem
    if extract_dir.exists():
        print(f"skip (already extracted): {extract_dir}")
        return extract_dir
    with zipfile.ZipFile(path) as zf:
        zf.extractall(extract_dir)
    return extract_dir


downloaded_paths = {}
for resource in resources:
    path = download_resource(resource)
    downloaded_paths[resource["name"]] = extract_if_zip(path)

downloaded_paths

skip (already downloaded): bikeshare-ridership-2014-2015.xlsx
skip (already downloaded): bikeshare-ridership-2016.xlsx
downloading bikeshare-ridership-2017.zip ...
downloading bikeshare-ridership-2018.zip ...
downloading bikeshare-ridership-2019.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2019
downloading bikeshare-ridership-2020.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2020
downloading bikeshare-ridership-2021.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2021
downloading bikeshare-ridership-2022.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2022
downloading bikeshare-ridership-2023.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2023
downloading bikeshare-ridership-2024.zip ...
downloading bikeshare-ridership-2025.zip ...
skip (already extracted): raw_data/bikeshare-ridership-2025
downloading bikeshare-ridership-2026.zip ...


{'bikeshare-ridership-2014-2015': PosixPath('raw_data/bikeshare-ridership-2014-2015.xlsx'),
 'bikeshare-ridership-2016': PosixPath('raw_data/bikeshare-ridership-2016.xlsx'),
 'bikeshare-ridership-2017.zip': PosixPath('raw_data/bikeshare-ridership-2017'),
 'bikeshare-ridership-2018.zip': PosixPath('raw_data/bikeshare-ridership-2018'),
 'bikeshare-ridership-2019.zip': PosixPath('raw_data/bikeshare-ridership-2019'),
 'bikeshare-ridership-2020.zip': PosixPath('raw_data/bikeshare-ridership-2020'),
 'bikeshare-ridership-2021.zip': PosixPath('raw_data/bikeshare-ridership-2021'),
 'bikeshare-ridership-2022.zip': PosixPath('raw_data/bikeshare-ridership-2022'),
 'bikeshare-ridership-2023.zip': PosixPath('raw_data/bikeshare-ridership-2023'),
 'bikeshare-ridership-2024.zip': PosixPath('raw_data/bikeshare-ridership-2024'),
 'bikeshare-ridership-2025.zip': PosixPath('raw_data/bikeshare-ridership-2025'),
 'bikeshare-ridership-2026.zip': PosixPath('raw_data/bikeshare-ridership-2026')}

# LOAD & CLEAN TRIPS

In [ ]:
import pandas as pd
import gc
from utils import load_year

DATA_DIR = Path("data")
CITY = "toronto"
YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
# 2014-2015's cleaning was left unfinished upstream (see import_data.ipynb) - excluded for now.

stations_vocab = pd.read_parquet("stations_vocab.parquet")
stations_actual_names = pd.read_parquet("stations_actual_names.parquet")

# station names are dropped in favor of station_id 
# Reindexing every year to this same column set keeps the partitioned parquet files
# schema-consistent even though older years are missing columns like bike_model.
TRIP_COLUMNS = [
    "trip_id", "start_time", "end_time", "trip_duration",
    "start_station_id", "end_station_id", "bike_id", "user_type", "bike_model",
]
CATEGORY_COLS = ["user_type", "bike_model"]

NUMERIC_DTYPES = {
    "trip_id": "Int64",
    "trip_duration": "Int32",
    "start_station_id": "Int32",
    "end_station_id": "Int32",
    "bike_id": "Int32",
}

def finalize_trip_columns(df):
    df = df.reindex(columns=TRIP_COLUMNS)
    for col in CATEGORY_COLS:
        # cast to string first - a year missing this column entirely (e.g. bike_model
        # before 2024) otherwise reindexes in as float64 NaN, giving that year's file a
        # float-backed category instead of string-backed, which breaks schema
        # unification when pyarrow reads the partitioned dataset back across years.
        df[col] = df[col].astype("string").astype("category")
    for col, dtype in NUMERIC_DTYPES.items():
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(dtype)
    return df

def trip_city_path(year):
    path = DATA_DIR / "trips" / f"city={CITY}" / f"year={year}"
    path.mkdir(parents=True, exist_ok=True)
    return path / "part.parquet"

In [11]:
trip_stats = {}
for year in YEARS:
    df = load_year(
        year,
        raw_dir=RAW_DATA_DIR,
        stations_vocab=stations_vocab,
        stations_actual_names=stations_actual_names,
    )
    start_missing_pct = df["start_station_id"].isna().mean() * 100
    end_missing_pct = df["end_station_id"].isna().mean() * 100

    df = finalize_trip_columns(df)
    df.to_parquet(trip_city_path(year), index=False, compression="zstd")

    trip_stats[year] = {
        "rows": len(df),
        "start_id_missing_pct": round(start_missing_pct, 3),
        "end_id_missing_pct": round(end_missing_pct, 3),
    }
    print(year, trip_stats[year])

    del df
    gc.collect()

pd.DataFrame(trip_stats).T

2016 {'rows': 367961, 'start_id_missing_pct': np.float64(0.001), 'end_id_missing_pct': np.float64(0.001)}
2017 {'rows': 1492367, 'start_id_missing_pct': np.float64(1.835), 'end_id_missing_pct': np.float64(1.966)}
2018 {'rows': 1922955, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.0)}
2019 {'rows': 2439517, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.019)}
2020 {'rows': 2911308, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.045)}
2021 {'rows': 3575182, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.055)}
2022 {'rows': 4620469, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.028)}
2023 {'rows': 5713141, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.052)}
2024 {'rows': 6953094, 'start_id_missing_pct': np.float64(0.0), 'end_id_missing_pct': np.float64(0.121)}
2025 {'rows': 7812520, 'start_id_missing_pct': np.floa

,rows,start_id_missing_pct,end_id_missing_pct
2016,367961.0,0.001,0.001
2017,1492367.0,1.835,1.966
2018,1922955.0,0.000,0.000
2019,2439517.0,0.000,0.019
2020,2911308.0,0.000,0.045
2021,3575182.0,0.000,0.055
2022,4620469.0,0.000,0.028
2023,5713141.0,0.000,0.052
2024,6953094.0,0.000,0.121
2025,7812520.0,0.000,0.012


In [12]:
trips24 = pd.read_parquet(DATA_DIR / "trips", filters=[("year", "=", 2024)])
print(trips24.shape)
print(trips24["year"].value_counts().sort_index())
trips24.dtypes

(6953094, 11)
year
2016          0
2017          0
2018          0
2019          0
2020          0
2021          0
2022          0
2023          0
2024    6953094
2025          0
2026          0
Name: count, dtype: int64


trip_id                      Int64
start_time          datetime64[us]
end_time            datetime64[us]
trip_duration                Int32
start_station_id             Int32
end_station_id               Int32
bike_id                      Int32
user_type                 category
bike_model                category
city                      category
year                      category
dtype: object

In [13]:
trips24.head()

,trip_id,start_time,end_time,trip_duration,start_station_id,end_station_id,bike_id,user_type,bike_model,city,year
0,26682740,2024-01-01 00:00:08,2024-01-01 00:08:56,528,7041,7130,7855,Casual,EFIT G5,toronto,2024
1,26682742,2024-01-01 00:02:04,2024-01-01 00:34:08,1924,7245,7259,2453,Casual,ICONIC,toronto,2024
2,26682743,2024-01-01 00:03:25,2024-01-01 00:49:06,2741,7052,7366,7332,Member,ICONIC,toronto,2024
3,26682744,2024-01-01 00:03:39,2024-01-01 00:09:32,353,7041,7418,782,Member,ICONIC,toronto,2024
4,26682746,2024-01-01 00:04:44,2024-01-01 00:40:02,2118,7721,7721,1008,Member,ICONIC,toronto,2024
